# Stage 1: get the data

**Reads:** the CEPII Gravity database, version V202211: `data/raw/Gravity_V202211.csv` if it is there, otherwise the CEPII zip in `data/raw/`, which is downloaded first if it is missing too.

**Writes:** `data/gravity_extract.csv`: every row of the development year (2017) and of the analysis year (2019), with the 16 columns the later stages use. The extract is committed. `data/raw/` is not: the zip (207 MB) and the CSV (1.25 GB) are over GitHub's 100 MB limit, so `.gitignore` leaves them out and this notebook is how they are obtained (Requirement 4.15).

Run it with `python code/run_all.py --download`. It needs about 1.5 GB of free disk space.

*Drafted with Claude (Anthropic), which was asked for the stage that downloads the CEPII Gravity file, checks that it is the file the project was planned on, and keeps the two chosen years with the columns the project uses.*

In [ ]:
import datetime
import hashlib
import shutil
import urllib.request
import zipfile

import pandas as pd

from helpers import ANALYSIS_YEAR, DATA, DEV_YEAR, RAW, ROOT

URL = "https://www.cepii.fr/DATA_DOWNLOAD/gravity/data/Gravity_csv_V202211.zip"
ZIP = RAW / "Gravity_csv_V202211.zip"
CSV = RAW / "Gravity_V202211.csv"
EXTRACT = DATA / "gravity_extract.csv"

YEARS = [DEV_YEAR, ANALYSIS_YEAR]
COLUMNS = [
    "year", "country_id_o", "country_id_d", "iso3_o", "iso3_d",
    "country_exists_o", "country_exists_d", "distw_harmonic",
    "contig", "comlang_off", "col_dep_ever", "comcol", "fta_wto",
    "gdp_o", "gdp_d", "tradeflow_baci",
]
# A row is one ordered pair of countries in one year. Countries are identified
# by country_id, not iso3: a few share an iso3 code, for example DEU.1 (West
# Germany) and DEU.2 (unified Germany).
KEY = ["year", "country_id_o", "country_id_d"]

# Read on Yu's copy of the file on 3 October 2026. A different value means the
# file is not the one the project was planned on: stop and report both values.
CSV_BYTES = 1_251_418_200
CSV_SHA256 = "c5611acb52288c537786dda86f2c3319cdd04cc94e8c9328dde936510a17c739"
ROWS_IN_FILE = 4_699_296
ROWS_PER_YEAR = 63_504  # 252 x 252 country identifiers, a country paired with itself included

## 1. Download and unpack

Each file is first written under a temporary `.part` name and renamed only when complete, so an interrupted run never leaves a file that looks finished. Running the notebook again starts that step over.

In [ ]:
def copy_to(source, target):
    """Copy a stream to target, through a .part file renamed at the end."""
    partial = target.with_name(target.name + ".part")
    with open(partial, "wb") as out:
        shutil.copyfileobj(source, out, 1 << 20)
    return partial


RAW.mkdir(parents=True, exist_ok=True)

if CSV.exists():
    print("Using the CSV already in data/raw/.")
else:
    if ZIP.exists():
        print("Using the zip already in data/raw/.")
    else:
        print("Downloading", URL)
        print("Date:", datetime.date.today().isoformat())
        request = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=60) as response:
            expected = response.headers.get("Content-Length")
            partial = copy_to(response, ZIP)
        if expected is not None:
            assert partial.stat().st_size == int(expected), "The download stopped early: run the notebook again."
        partial.replace(ZIP)
        print(f"Downloaded {ZIP.stat().st_size / 1e6:.0f} MB")

    with zipfile.ZipFile(ZIP) as archive:
        names = archive.namelist()
        print("Files in the zip:", names)
        # The folder inside the zip is not assumed: take the file with the expected name.
        matches = [name for name in names if name.rsplit("/", 1)[-1] == CSV.name]
        assert len(matches) == 1, f"Expected one {CSV.name} in the zip, found {matches}"
        with archive.open(matches[0]) as member:
            copy_to(member, CSV).replace(CSV)
    print("Unpacked", CSV.name)

## 2. Check that it is the file the project was planned on

Its size and SHA-256 fingerprint are compared with those of Yu's copy, read on 3 October 2026. If CEPII has changed the file, the run stops here, because every count in the project was read on that copy.

In [ ]:
size = CSV.stat().st_size
with open(CSV, "rb") as file:
    sha256 = hashlib.file_digest(file, "sha256").hexdigest()

print(f"Size:    {size:,} bytes, expected {CSV_BYTES:,}")
print(f"SHA-256: {sha256}")
print(f"         {CSV_SHA256} expected")
assert size == CSV_BYTES and sha256 == CSV_SHA256, "Not the file the project was planned on: stop and report both values."

## 3. Keep the two years and the 16 columns

The file is read in chunks of 250,000 rows, so the 1.25 GB are never in memory at once. Every value is read as text and written back unchanged, so the extract holds exactly what the CEPII file holds: types are set in stage 2, and an empty field (a missing value) stays empty.

In [ ]:
wanted = {str(year) for year in YEARS}
rows_read = 0
kept = []
for chunk in pd.read_csv(CSV, usecols=COLUMNS, dtype=str, keep_default_na=False, chunksize=250_000):
    rows_read += len(chunk)
    kept.append(chunk[chunk["year"].isin(wanted)])
extract = pd.concat(kept, ignore_index=True)[COLUMNS]

print(f"Rows read: {rows_read:,}")
print(f"Rows kept: {len(extract):,}")

## 4. Checks

In [ ]:
per_year = extract["year"].value_counts().sort_index()
print("Rows per year:")
print(per_year.to_string())

assert rows_read == ROWS_IN_FILE, f"{rows_read:,} rows read, {ROWS_IN_FILE:,} expected"
assert per_year.to_dict() == {str(year): ROWS_PER_YEAR for year in YEARS}, f"{ROWS_PER_YEAR:,} rows per year expected"
assert list(extract.columns) == COLUMNS
assert not extract.duplicated(KEY).any(), "Some (year, origin, destination) appears twice"
print("Checks passed: rows read, rows per year, columns, no repeated key.")

## 5. Save the extract

The extract is committed, so the later stages run without the download. Its SHA-256 fingerprint is printed so that a new run can be compared with the committed copy (`data/README.md`): the same file from CEPII gives the same extract.

In [ ]:
extract.to_csv(EXTRACT, index=False, lineterminator="\n")
with open(EXTRACT, "rb") as file:
    extract_sha256 = hashlib.file_digest(file, "sha256").hexdigest()

print(f"Saved {EXTRACT.relative_to(ROOT)}: {len(extract):,} rows, {len(COLUMNS)} columns, {EXTRACT.stat().st_size / 1e6:.1f} MB")
print("SHA-256:", extract_sha256)